In [1]:
import xarray as xr
import rioxarray
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score
from scipy.ndimage import median_filter, minimum_filter, maximum_filter, uniform_filter
import lightgbm as lgb 
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import joblib

In [2]:
payload = joblib.load('trained.joblib')
X_train = payload['X_train']
y_train = payload['y_train']
X_test = payload['X_test']
y_test = payload['y_test']
clf = payload['clf']

X_train.head(1)

,blue,green,red,rededge,B06,B07,nir,B8A,swir11,swir22,...,ndwi__5x5_median,tci__5x5_var,ndvi__5x5_var,evi__5x5_var,evi2__5x5_var,ndre_b7__5x5_var,ndre_b6__5x5_var,ndre_b5__5x5_var,gndvi__5x5_var,ndwi__5x5_var
48210,0.027451,0.035294,0.039216,0.105882,0.082353,0.090196,0.066667,0.090196,0.141176,0.105882,...,-0.272727,-0.005483,0.8125,0.215232,0.348837,0.025641,0.25,0.95122,0.223214,0.348837


# Real area from Cocoa area estimation

My issue is that I'm dealing with 2 back-to-back area estimation.
Hence, for each region, I have

1. The real cocoa area
2. The ETHZ's cocoa area estimation
3. My cocoa area estimation

First, I need to figure out how to estimate the real cocoa area from the ETHZ's cocoa area estimation.

## ETHZ's estimate

### Naive Area Estimate

$$A_{ETHZ} = Positive_{ETHZ} = TP_{ETHZ} + FP_{ETHZ}$$

$$Â^{naive}_{ETHZ} = A_{ETHZ}$$


In [3]:
# Cocoa area according to the ETHZ, naively

def compute_naive_area_m2(y) -> float:
    """Compute the naive area (i.e. pixel counting) given a list of prediction."""
    M2_PER_PX = 10*10
    return (y).sum() * M2_PER_PX

area_train = compute_naive_area_m2(y_train)
area_test = compute_naive_area_m2(y_test)

print(f"Cocoa area:")
print(f"- Train set")
print(f"--- ETHZ {area_train}m2")
print(f"- Test set")
print(f"--- ETHZ {area_test}m2")

Cocoa area:
- Train set
--- ETHZ 238300m2
- Test set
--- ETHZ 21500m2


### Bias correction

$$A_{ETHZ} = Positive_{ETHZ} = TP_{ETHZ} + FP_{ETHZ} = \frac{TP_{ETHZ}}{Precision_{ETHZ}}$$
$$A_{true} = TP_{ETHZ} + FN_{ETHZ} = \frac{TP_{ETHZ}}{Recall_{ETHZ}}$$

$$\rightarrow A_{true} = A_{ETHZ} * \frac{Precision_{ETHZ}}{Recall_{ETHZ}}$$

$A_{true}$ is the actual cocoa area.\
The thing is, our precision/recall is estimated off some subset (i.e. our test set).\
Hence we make the assumption that our test set's Precision/Recall matches the actual Precision/Recall of our model.

$$\rightarrow Â^{corrected}_{ETHZ} = A_{ETHZ} * \frac{Precision^{test}_{ETHZ}}{Recall^{test}_{ETHZ}}$$

In [4]:
def compute_corrected_area_m2(y, precision, recall):
    M2_PER_PX = 10*10 
    area_m2 = y.sum() * M2_PER_PX
    return area_m2 * precision / recall

M2_PER_PX = 10*10 
total_train_m2 = len(y_train) * M2_PER_PX
total_test_m2 = len(y_test) * M2_PER_PX

area_train = compute_corrected_area_m2(y_train, precision=0.917, recall=0.909) # Precision/Recall from ETHZ stufy
area_test = compute_corrected_area_m2(y_test, precision=0.917, recall=0.909)

print(f"Cocoa area:")
print(f"- Train set")
print(f"--- ETHZ {area_train / total_train_m2 * 100:.3f}% of total area ({area_train:.0f}m2)")
print(f"- Test set")
print(f"--- ETHZ {area_test / total_test_m2 * 100:.3f}% of total area ({area_test:.0f}m2)")

Cocoa area:
- Train set
--- ETHZ 26.755% of total area (240397m2)
- Test set
--- ETHZ 21.453% of total area (21689m2)


## Our estimate

The ETHZ's estimated area are the best we'll get.
We make the assumption that the ETHZ's estimate is correct, i.e

$$Â^{corrected}_{ETHZ} = A_{true}$$

From them, we'll compare our estimate, i.e.

$$A_{true} = Â^{corrected}_{ETHZ} = A_{LGBM} * \frac{Precision_{LGBM}}{Recall_{LGBM}}$$
$$\rightarrow Â^{corrected}_{LGBM} = A_{LGBM} * \frac{Precision^{test}_{LGBM}}{Recall^{test}_{LGBM}}$$

We'll measure how far off $Â^{corrected}_{ETHZ}$ we are

In [5]:
def compute_corrected_area_m2(y, precision, recall):
    M2_PER_PX = 10*10 
    area_m2 = y.sum() * M2_PER_PX
    return area_m2 * precision / recall

M2_PER_PX = 10*10 
total_train_m2 = len(y_train) * M2_PER_PX
total_test_m2 = len(y_test) * M2_PER_PX

area_ethz_train = compute_corrected_area_m2(y_train, precision=0.917, recall=0.909) # Precision/Recall from ETHZ stufy
area_ethz_test = compute_corrected_area_m2(y_test, precision=0.917, recall=0.909)

# Compute test set's precision & recall
yhat_train = clf.predict(X_train)
yhat_test = clf.predict(X_test)
precision = precision_score(y_test, yhat_test)
recall = recall_score(y_test, yhat_test)

area_train = compute_corrected_area_m2(yhat_train, precision, recall)
area_test = compute_corrected_area_m2(yhat_test, precision, recall)

print(f"Cocoa area:")
print(f"- Train set")
print(f"--- Our {area_train / total_train_m2 * 100:.2f}% of total area ({area_train:.0f}m2, {area_train/area_ethz_train * 100 - 100:.2f}% off ETHZ's estimate)")
print(f"- Test set")
print(f"--- Our {area_test / total_test_m2 * 100:.2f}% of total area ({area_test:.0f}m2, {area_test/area_ethz_test * 100 - 100:.2f}% off ETHZ's estimate)")

Cocoa area:
- Train set
--- Our 26.97% of total area (242311m2, 0.80% off ETHZ's estimate)
- Test set
--- Our 21.27% of total area (21500m2, -0.87% off ETHZ's estimate)
